# CHECKPOINT 2.6 — Authentication & Zero-Trust RBAC Verification Notebook

This notebook verifies the Firebase Authentication, Firestore profile linking, and Role-Based Access Control (RBAC) implementation for SmartEdu ERP.

In [ ]:
import json

# Define RBAC Mapping for Verification
ALLOWED_TABS_BY_ROLE = {
    'ADMIN': ['dashboard', 'academic', 'students', 'teachers', 'parents', 'classes', 'homework', 'scores', 'risk_ai', 'tuition', 'audit_logs', 'settings'],
    'ACADEMIC_STAFF': ['dashboard', 'academic', 'students', 'teachers', 'parents', 'classes', 'homework', 'scores', 'risk_ai'],
    'ACCOUNTANT': ['dashboard', 'tuition'],
    'TEACHER': ['dashboard', 'classes', 'homework', 'scores', 'students'],
    'STUDENT': ['dashboard', 'scores', 'homework', 'tuition'],
    'PARENT': ['dashboard', 'scores', 'homework', 'tuition']
}

print("RBAC Profile Rules Loaded Successfully.")
for role, tabs in ALLOWED_TABS_BY_ROLE.items():
    print(f"Role: {role:<15} -> Accessible Tabs: {len(tabs)}")

In [ ]:
def verify_auth_policy(user_profile):
    if not user_profile:
        return False, "[AUTH_PROFILE_NOT_FOUND] Denied - Missing Firestore profile doc"
    
    status = user_profile.get('status', '')
    if status not in ['Đang hoạt động', 'ACTIVE']:
        return False, f"[AUTH_PROFILE_INACTIVE] Denied - Profile status is '{status}'"
        
    role = user_profile.get('role', '')
    if role not in ALLOWED_TABS_BY_ROLE:
        return False, f"[AUTH_INVALID_ROLE] Denied - Unknown role '{role}'"
        
    return True, f"[AUTH_SUCCESS] Access granted for role '{role}'"

# Test Cases
test_cases = [
    {"name": "Valid Admin", "profile": {"id": "uid_admin", "role": "ADMIN", "status": "Đang hoạt động"}},
    {"name": "Missing Profile", "profile": None},
    {"name": "Inactive Account", "profile": {"id": "uid_locked", "role": "TEACHER", "status": "Bị khóa"}},
    {"name": "Invalid Role", "profile": {"id": "uid_hacker", "role": "SUPERUSER", "status": "Đang hoạt động"}}
]

for tc in test_cases:
    allowed, msg = verify_auth_policy(tc['profile'])
    status_str = "PASS" if allowed else "BLOCKED"
    print(f"Test '{tc['name']}': [{status_str}] - {msg}")